# Explainable Machine-Learning and Predictive Statistics for Sustainable Photovoltaic Power Prediction on Areal Meteorological Variables
Sajjad Nematzadeh, 
Vedat Esen

In [2]:
from mealpy import BoolVar, GA
import numpy as np
import pandas as pd
from sklearn import tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import Helper
from sklearn.preprocessing import MinMaxScaler
import torch

In [4]:
df_result = {'cols': [], 'features_count': [], 'score': [],
             'train_mse': [], 'val_mse': [], 'test_mse': [],
             'train_r2': [], 'val_r2': [], 'test_r2': []}
performance_coefficient = 0.95


def load_data(to_torch=False):
    dfo = pd.read_csv(fr'data/apw.csv', index_col=None)
    X_cols = ['temperature_2m', 'relative_humidity_2m', 'apparent_temperature',
              'wind_speed_10m', 'wind_direction_10m',
              'shortwave_radiation_instant', 'direct_radiation_instant', 'diffuse_radiation_instant',
              'direct_normal_irradiance',
              'direct_normal_irradiance_instant', 'global_tilted_irradiance_instant', 'terrestrial_radiation_instant']

    X = dfo.loc[:, X_cols]
    y = dfo['AP']

    scalerX = MinMaxScaler()
    scalerX.fit(X)
    X = pd.DataFrame(scalerX.transform(X), columns=X.columns)

    y = y.values.reshape(-1, 1)
    scalerY = MinMaxScaler()
    scalerY.fit(y)
    y = scalerY.transform(y)

    if to_torch:
        y = torch.FloatTensor(y).view(-1, 1)
        X = torch.FloatTensor(X.values)
    return X, y


x, y = load_data(to_torch=True)


def objective_function(cols):
    global df_result
    cols = np.array(cols).astype(bool)
    number_of_features = sum(cols)
    if number_of_features == 0:
        return 10

    data = (x, y)
    mlp = Helper.MLP(data, val_size=0.2, test_size=0.2, epochs=25, batch_size=1000)
    history = mlp.train_model()
    test_mse, test_r2 = mlp.test_model()

    df_result['cols'] += [''.join(cols.astype(int).astype(str))]
    df_result['features_count'] += [number_of_features]
    df_result['train_mse'] += [history[-1]['train_loss']]
    df_result['val_mse'] += [history[-1]['val_loss']]
    df_result['test_mse'] += [test_mse]
    df_result['train_r2'] += [history[-1]['train_r2']]
    df_result['val_r2'] += [history[-1]['val_r2']]
    df_result['test_r2'] += [test_r2]
    score = -test_r2*performance_coefficient+(number_of_features/len(cols))*(1-performance_coefficient)
    df_result['score'] += [score]
    return score

In [5]:
problem_dict = {
    "bounds": BoolVar(x.shape[1], name="delta"),
    "obj_func": objective_function,
    "minmax": "min",
}

In [6]:
model = GA.BaseGA(epoch=30, pop_size=30, pc=0.8, pm=0.2)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

Epoch [10/25], Train MSE: 0.0064, Train R2: 0.8950, Val MSE: 0.0059, Val R2: 0.9015
Epoch [20/25], Train MSE: 0.0060, Train R2: 0.9019, Val MSE: 0.0055, Val R2: 0.9088


2025/01/24 05:51:09 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: Solving single objective optimization problem.


Test MSE: 0.0057, Test R2: 0.9066
Epoch [10/25], Train MSE: 0.0077, Train R2: 0.8735, Val MSE: 0.0072, Val R2: 0.8796
Epoch [20/25], Train MSE: 0.0068, Train R2: 0.8890, Val MSE: 0.0063, Val R2: 0.8957
Test MSE: 0.0063, Test R2: 0.8955
Epoch [10/25], Train MSE: 0.0083, Train R2: 0.8633, Val MSE: 0.0079, Val R2: 0.8692
Epoch [20/25], Train MSE: 0.0072, Train R2: 0.8821, Val MSE: 0.0067, Val R2: 0.8885
Test MSE: 0.0066, Test R2: 0.8916
Epoch [10/25], Train MSE: 0.0069, Train R2: 0.8866, Val MSE: 0.0064, Val R2: 0.8929
Epoch [20/25], Train MSE: 0.0062, Train R2: 0.8990, Val MSE: 0.0057, Val R2: 0.9056
Test MSE: 0.0057, Test R2: 0.9054
Epoch [10/25], Train MSE: 0.0062, Train R2: 0.8975, Val MSE: 0.0057, Val R2: 0.9048
Epoch [20/25], Train MSE: 0.0060, Train R2: 0.9018, Val MSE: 0.0055, Val R2: 0.9092
Test MSE: 0.0057, Test R2: 0.9055
Epoch [10/25], Train MSE: 0.0075, Train R2: 0.8778, Val MSE: 0.0069, Val R2: 0.8846
Epoch [20/25], Train MSE: 0.0068, Train R2: 0.8883, Val MSE: 0.0063, Val R

2025/01/24 06:11:56 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: >>>Problem: P, Epoch: 1, Current best: -0.8468912542241551, Global best: -0.8468912542241551, Runtime: 635.17837 seconds


Test MSE: 0.0068, Test R2: 0.8877
Epoch [10/25], Train MSE: 0.0068, Train R2: 0.8888, Val MSE: 0.0063, Val R2: 0.8952
Epoch [20/25], Train MSE: 0.0064, Train R2: 0.8950, Val MSE: 0.0059, Val R2: 0.9019
Test MSE: 0.0061, Test R2: 0.8998
Epoch [10/25], Train MSE: 0.0073, Train R2: 0.8808, Val MSE: 0.0068, Val R2: 0.8872
Epoch [20/25], Train MSE: 0.0064, Train R2: 0.8954, Val MSE: 0.0059, Val R2: 0.9022
Test MSE: 0.0059, Test R2: 0.9025
Epoch [10/25], Train MSE: 0.0065, Train R2: 0.8940, Val MSE: 0.0060, Val R2: 0.9004
Epoch [20/25], Train MSE: 0.0062, Train R2: 0.8990, Val MSE: 0.0057, Val R2: 0.9057
Test MSE: 0.0059, Test R2: 0.9031
Epoch [10/25], Train MSE: 0.0076, Train R2: 0.8752, Val MSE: 0.0071, Val R2: 0.8824
Epoch [20/25], Train MSE: 0.0068, Train R2: 0.8878, Val MSE: 0.0063, Val R2: 0.8949
Test MSE: 0.0064, Test R2: 0.8949
Epoch [10/25], Train MSE: 0.0077, Train R2: 0.8745, Val MSE: 0.0071, Val R2: 0.8810
Epoch [20/25], Train MSE: 0.0069, Train R2: 0.8863, Val MSE: 0.0064, Val R

2025/01/24 06:22:31 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: >>>Problem: P, Epoch: 2, Current best: -0.8496412416285868, Global best: -0.8496412416285868, Runtime: 634.88693 seconds


... (53 outputs left out)

2025/01/24 10:52:48 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: >>>Problem: P, Epoch: 29, Current best: -0.850383908570297, Global best: -0.8596402777827792, Runtime: 591.21622 seconds


Test MSE: 0.0058, Test R2: 0.9048
Epoch [10/25], Train MSE: 0.0064, Train R2: 0.8954, Val MSE: 0.0059, Val R2: 0.9018
Epoch [20/25], Train MSE: 0.0061, Train R2: 0.9007, Val MSE: 0.0056, Val R2: 0.9072
Test MSE: 0.0057, Test R2: 0.9058
Epoch [10/25], Train MSE: 0.0089, Train R2: 0.8538, Val MSE: 0.0085, Val R2: 0.8586
Epoch [20/25], Train MSE: 0.0076, Train R2: 0.8754, Val MSE: 0.0071, Val R2: 0.8810
Test MSE: 0.0070, Test R2: 0.8852
Epoch [10/25], Train MSE: 0.0063, Train R2: 0.8968, Val MSE: 0.0058, Val R2: 0.9030
Epoch [20/25], Train MSE: 0.0060, Train R2: 0.9015, Val MSE: 0.0055, Val R2: 0.9081
Test MSE: 0.0057, Test R2: 0.9060
Epoch [10/25], Train MSE: 0.0065, Train R2: 0.8939, Val MSE: 0.0060, Val R2: 0.9003
Epoch [20/25], Train MSE: 0.0060, Train R2: 0.9015, Val MSE: 0.0055, Val R2: 0.9083
Test MSE: 0.0057, Test R2: 0.9065
Epoch [10/25], Train MSE: 0.0074, Train R2: 0.8786, Val MSE: 0.0069, Val R2: 0.8851
Epoch [20/25], Train MSE: 0.0068, Train R2: 0.8879, Val MSE: 0.0063, Val R

2025/01/24 11:03:24 PM, INFO, mealpy.evolutionary_based.GA.BaseGA: >>>Problem: P, Epoch: 30, Current best: -0.8565136131897237, Global best: -0.8596402777827792, Runtime: 635.05637 seconds


Test MSE: 0.0061, Test R2: 0.8996
Solution: [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.], Fitness: -0.8596402777827792
Solution: [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.], Fitness: -0.8596402777827792


In [7]:
df_result = pd.DataFrame(df_result)
df_result.to_csv('data/meta_h_mlp_95.csv')

In [6]:
df_result = pd.read_csv('data/meta_h_mlp_95.csv', index_col=None)
df_result.sort_values(['score'])

,index,cols,features_count,score,train_mse,val_mse,test_mse,train_r2,val_r2,test_r2
807,807,10000,1,-0.859640,0.005753,0.005252,0.005514,0.905624,0.912536,0.909270
612,612,1000000000,1,-0.859638,0.005688,0.005208,0.005514,0.906686,0.913260,0.909268
129,129,100000000000,1,-0.858391,0.005761,0.005291,0.005594,0.905494,0.911880,0.907955
709,709,100000000,1,-0.858008,0.005811,0.005333,0.005618,0.904669,0.911179,0.907552
762,762,1000000000,1,-0.857681,0.005810,0.005297,0.005639,0.904691,0.911785,0.907208
...,...,...,...,...,...,...,...,...,...,...
87,87,10011011111,8,-0.807506,0.007198,0.006662,0.006983,0.881919,0.889044,0.885094
629,629,1110101000,5,-0.806178,0.008102,0.007673,0.007868,0.867081,0.872216,0.870538
33,33,111111111101,11,-0.798584,0.006976,0.006501,0.006754,0.885560,0.891739,0.888861
14,14,111110111000,8,-0.790465,0.008295,0.007830,0.008073,0.863914,0.869598,0.867156
